# 🔍 Bing Grounding Agent — Solution

This notebook provides a complete working solution to the Bing Grounding challenge.
The agent researches Morocco's digital economy using live Bing search.

In [ ]:
# ! pip install -r ../../../../requirements.txt -U

In [ ]:
from agent_framework.azure import AzureAIAgentClient, AzureAIAgentsProvider
from azure.identity import AzureCliCredential

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
)
from azure.core.exceptions import ResourceNotFoundError

In [ ]:
import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv("../../../../.env")

In [ ]:
AgentName = "Morocco-BizIntel-Agent"

# SOLUTION — Task 1: Agent Instructions
AgentInstructions = """You are a Morocco Business Intelligence research assistant.
Your role is to help investors and business professionals understand Morocco's
digital economy, tech sector, and business environment.

- Always use the Bing search tool to find current, up-to-date information.
- Cite your sources clearly for every key fact.
- Provide structured, actionable insights.
- At the end of your answer, add one interesting business fact about Casablanca."""

In [ ]:
env_connection_name = os.environ["BING_CONNECTION_NAME"]

with AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
) as project:
    bing_connection = None
    try:
        bing_connection = project.connections.get(env_connection_name)
        print(f"Resolved connection: {bing_connection.name} -> {bing_connection.id}")
    except ResourceNotFoundError:
        pass

    if bing_connection is None:
        available = [c.name for c in project.connections.list()]
        raise RuntimeError(
            f"No matching Bing connection found. Tried: {env_connection_name}. "
            f"Available project connections: {available}"
        )

    # SOLUTION — Task 2: Version management + Configure BingGroundingTool and create agent
    agent_name = AgentName
    latest_version = None

    try:
        versions = list(project.agents.list_versions(agent_name=agent_name))
        if versions:
            latest_agent = versions[0]
            latest_version = int(latest_agent.version) if hasattr(latest_agent, 'version') else latest_agent.version
            print(f"Agent '{agent_name}' exists. Latest version: {latest_version}.")
    except ResourceNotFoundError:
        print(f"Agent '{agent_name}' does not exist. A new agent will be created.")

    model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME")

    agent = project.agents.create_version(
        agent_name=AgentName,
        definition=PromptAgentDefinition(
            model=model_name,
            instructions=AgentInstructions,
            tools=[
                BingGroundingTool(
                    bing_grounding=BingGroundingSearchToolParameters(
                        search_configurations=[
                            BingGroundingSearchConfiguration(
                                project_connection_id=bing_connection.id
                            )
                        ]
                    )
                )
            ],
        ),
        description="Morocco business intelligence agent with Bing grounding",
    )

    if latest_version is not None and int(agent.version) == int(latest_version):
        print(f"No changes detected, agent version remains the same: {agent.version}")
    else:
        print(f"\n✓ Agent version {agent.version} created successfully")
        print(f"  - Agent ID: {agent.id}")
        print(f"  - Agent Name: {agent.name}")
        print(f"  - Version: {agent.version}")
        if latest_version:
            print(f"  - Previous version: {latest_version}")

In [ ]:
# SOLUTION — Task 3: A question about Morocco's digital economy
UserQuestion = "What is Morocco's national digital transformation strategy and what progress has been made recently?"

In [ ]:
# SOLUTION — Task 4: Query the agent
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
)

openai_client = project_client.get_openai_client()

response = openai_client.responses.create(
    input=[{"role": "user", "content": UserQuestion}],
    extra_body={"agent_reference": {"name": AgentName, "version": agent.version, "type": "agent_reference"}},
)

print(f"Response output: {response.output_text}")